# Exchangeability Plotting Notebook
Generate publication-style plots from an exchangeability CSV.

In [ ]:
import os
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
from scripts.plot_exchangeability import _prepare, _aggregate_for_curves, _plot_metric, _plot_train_val, _plot_within_observed_significance

In [ ]:
BASE_SAVE_DIR = Path(os.environ.get('BASE_SAVE_DIR', '/n/netscratch/kempner_pehlevan_lab/Lab/ilavie/exchangeability_outputs'))
CSV_OVERRIDE = os.environ.get('EXCHANGEABILITY_CSV', '').strip()
REQUIRED_ANALYSIS_COLUMNS = {
    'width',
    'images_seen',
    'representation',
    'analysis_type',
    'shuffle_id',
    'ks_distance',
    'w1_distance',
}

# KS/W1 curve controls (exposed for quick notebook edits)
KS_W1_ANALYSIS_TYPES = [
    'within_vs_across_real',
    'within_shuffled_vs_across_real',
]
KS_W1_ANALYSIS_LABELS = {
    'within_vs_across_real': 'within-across',
    'within_shuffled_vs_across_real': 'within (shuffled)-across (real)',
}
# Per-width color pairing: full color for within-across, lighter shade for within (shuffled)-across (real)
KS_W1_ANALYSIS_COLOR_ADJUST = {
    'within_vs_across_real': 0.0,
    'within_shuffled_vs_across_real': 0.45,
}
KS_W1_REPRESENTATION_LINESTYLES = {
    'weights': '-',
    'activations': '--',
    'activation': '--',
}
KS_W1_REPRESENTATION_ORDER = ['weights', 'activations', 'activation']

if CSV_OVERRIDE:
    candidates = [Path(CSV_OVERRIDE)]
else:
    preferred = [
        BASE_SAVE_DIR / 'exchangeability_metrics.csv',
        Path('outputs') / 'exchangeability_metrics.csv',
    ]
    all_candidates = list(BASE_SAVE_DIR.glob('exchangeability_metrics*.csv')) + list(Path('outputs').glob('exchangeability_metrics*.csv'))
    fallback = sorted(
        [p for p in all_candidates if '_summary' not in p.stem],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    candidates = []
    seen = set()
    for p in preferred + fallback:
        key = str(p)
        if key in seen:
            continue
        seen.add(key)
        candidates.append(p)

CSV_PATH = None
skipped = []
for p in candidates:
    if not p.exists():
        continue
    try:
        columns = set(pd.read_csv(p, nrows=0).columns.astype(str))
    except Exception as exc:
        skipped.append((p, f'unreadable header: {exc}'))
        continue
    missing = sorted(REQUIRED_ANALYSIS_COLUMNS - columns)
    if missing:
        skipped.append((p, f'missing columns: {missing}'))
        continue
    CSV_PATH = p
    break

if CSV_PATH is None:
    preview = '\n'.join([f'  - {p}: {reason}' for p, reason in skipped[:8]])
    raise FileNotFoundError(
        'No compatible raw exchangeability metrics CSV found (summary files are not valid for plotting).\n'
        + preview
    )
for p, reason in skipped[:3]:
    print(f'Skipping CSV candidate {p}: {reason}')
OUT_DIR = CSV_PATH.parent / f'plots_{CSV_PATH.stem}_notebook'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Using CSV_PATH={CSV_PATH}')
print(f'Writing OUT_DIR={OUT_DIR}')
print(f'KS/W1 analysis types: {KS_W1_ANALYSIS_TYPES}')
df = _prepare(pd.read_csv(CSV_PATH))
available_widths = sorted(df['width'].astype(int).unique().tolist())
print(f'Available widths in CSV: {available_widths}')
cmap = plt.get_cmap('tab10')
KS_W1_WIDTH_COLORS = {int(w): cmap(i % cmap.N) for i, w in enumerate(available_widths)}
print(f'KS/W1 width colors: {KS_W1_WIDTH_COLORS}')
curves = _aggregate_for_curves(df)


In [ ]:
fig_ks = _plot_metric(
    curves,
    metric='ks_distance',
    lo='ks_distance_p10',
    hi='ks_distance_p90',
    out_path=str(OUT_DIR / 'ks_distance_vs_images_seen.pdf'),
    title='KS Distance vs Images Seen',
    close=False,
    analysis_types=KS_W1_ANALYSIS_TYPES,
    analysis_labels=KS_W1_ANALYSIS_LABELS,
    width_colors=KS_W1_WIDTH_COLORS,
    analysis_color_adjust=KS_W1_ANALYSIS_COLOR_ADJUST,
    representation_linestyles=KS_W1_REPRESENTATION_LINESTYLES,
    representation_order=KS_W1_REPRESENTATION_ORDER,
)
display(fig_ks)

fig_w1 = _plot_metric(
    curves,
    metric='w1_distance',
    lo='w1_distance_p10',
    hi='w1_distance_p90',
    out_path=str(OUT_DIR / 'w1_distance_vs_images_seen.pdf'),
    title='W1 Distance vs Images Seen',
    close=False,
    analysis_types=KS_W1_ANALYSIS_TYPES,
    analysis_labels=KS_W1_ANALYSIS_LABELS,
    width_colors=KS_W1_WIDTH_COLORS,
    analysis_color_adjust=KS_W1_ANALYSIS_COLOR_ADJUST,
    representation_linestyles=KS_W1_REPRESENTATION_LINESTYLES,
    representation_order=KS_W1_REPRESENTATION_ORDER,
)
display(fig_w1)

for fig in _plot_train_val(df, str(OUT_DIR), close=False):
    display(fig)


for fig in _plot_within_observed_significance(
    df,
    str(OUT_DIR),
    close=False,
    width_colors=KS_W1_WIDTH_COLORS,
    representation_linestyles=KS_W1_REPRESENTATION_LINESTYLES,
    representation_order=KS_W1_REPRESENTATION_ORDER,
):
    display(fig)
plt.close('all')
print(f'Wrote plots to {OUT_DIR}')


In [ ]:
import numpy as np
import matplotlib.colors as mcolors
from scripts.analyze_exchangeability import (
    _resolve_width_dirs,
    _list_group_dirs,
    _collect_target_steps,
    _missing_weight_artifacts,
    _extract_weights_from_artifacts,
    _weight_similarity_matrix,
    _collect_member_states,
    _activation_similarity_matrix,
    _build_probe_loader,
)
from src.experiment.exchangeability_utils import (
    build_member_ids,
    extract_across_values,
    extract_within_values,
)

ECDF_DATA_DIR = CSV_PATH.with_name(CSV_PATH.stem + '_similarity')
ECDF_RUN_ID = 'exchangeability'
ECDF_RUN_ID_RESOLUTION = 'latest_prefix'
AVAILABLE_WIDTHS = sorted(df['width'].astype(int).unique().tolist())
ECDF_WIDTH = int(AVAILABLE_WIDTHS[0]) if AVAILABLE_WIDTHS else 32
ECDF_REPRESENTATION = 'weights'  # 'weights' or 'activations'
ECDF_MAX_POINTS = 200000
ECDF_STEPS = None  # set list[int] to restrict specific P values
ECDF_RNG_SEED = 0

# Used only when fallback compute is needed and ECDF_REPRESENTATION='activations'
ECDF_PROBE_BATCH_SIZE = 1024
ECDF_PROBE_LOADER_BATCH_SIZE = 128
ECDF_PROBE_SEED = 1234

print(f'ECDF_DATA_DIR={ECDF_DATA_DIR}')


In [ ]:
def _ecdf_xy(values: np.ndarray):
    xs = np.sort(values)
    ys = np.arange(1, xs.size + 1, dtype=np.float64) / float(xs.size)
    return xs, ys


def _subsample(values: np.ndarray, max_points: int, rng: np.random.Generator):
    if max_points <= 0 or values.size <= max_points:
        return values
    idx = rng.choice(values.size, size=max_points, replace=False)
    return values[idx]


def _lighten(color, amount: float):
    rgba = np.array(mcolors.to_rgba(color), dtype=np.float64)
    white = np.array([1.0, 1.0, 1.0, rgba[3]], dtype=np.float64)
    mixed = (1.0 - amount) * rgba + amount * white
    return tuple(mixed.tolist())


width_dir = ECDF_DATA_DIR / f'width_{ECDF_WIDTH}'
available_steps = []
for step_dir in sorted(width_dir.glob('step_*')):
    try:
        step = int(step_dir.name.split('_')[-1])
    except ValueError:
        continue
    npz_path = step_dir / f'{ECDF_REPRESENTATION}.npz'
    if npz_path.exists():
        available_steps.append(step)

width_mask = df['width'].astype(int) == int(ECDF_WIDTH)
rep_mask = df['representation'].astype(str) == str(ECDF_REPRESENTATION)
csv_steps = sorted({int(s) for s in df.loc[width_mask & rep_mask, 'images_seen'].tolist()})

can_compute = False
group_dirs = []
target_steps = []
try:
    width_dirs, width_sources = _resolve_width_dirs(
        str(BASE_SAVE_DIR),
        ECDF_RUN_ID,
        ECDF_RUN_ID_RESOLUTION,
        requested_widths=[int(ECDF_WIDTH)],
    )
    if ECDF_WIDTH in width_dirs:
        resolved_run_id = width_sources.get(ECDF_WIDTH, '')
        print(f'Fallback compute resolved run_id={resolved_run_id} for width={ECDF_WIDTH}')
        group_dirs = _list_group_dirs(width_dirs[ECDF_WIDTH])
        if group_dirs:
            target_steps = _collect_target_steps(group_dirs)
            can_compute = True
except Exception as exc:
    print(f'Could not resolve run data for fallback computation: {exc}')

target_step_set = set(target_steps)

if ECDF_STEPS is None:
    steps = sorted(set(available_steps) | set(csv_steps) | target_step_set)
else:
    steps = [int(s) for s in ECDF_STEPS]

if not steps:
    raise ValueError(
        f'No ECDF candidate steps for width={ECDF_WIDTH}, representation={ECDF_REPRESENTATION}. '
        f'cache_steps={len(available_steps)}, csv_steps={len(csv_steps)}, compute_steps={len(target_steps)}. '
        f'Expected cache under {width_dir}.'
    )

print(
    f'ECDF step sources: cache={len(available_steps)} csv={len(csv_steps)} '
    f'compute={len(target_steps)} selected={len(steps)}'
)

probe_loader = None
ecdf_data = {}
for step in steps:
    npz_path = width_dir / f'step_{int(step)}' / f'{ECDF_REPRESENTATION}.npz'

    if npz_path.exists():
        with np.load(npz_path) as arr:
            within_real = np.asarray(arr['within_real'])
            across_real = np.asarray(arr['across_real'])
    else:
        if (not can_compute) or (int(step) not in target_step_set):
            print(f'Skipping P={step}: cache missing and fallback compute unavailable for this step.')
            continue

        if ECDF_REPRESENTATION == 'weights':
            missing = _missing_weight_artifacts(group_dirs, int(step))
            if missing:
                print(f'Skipping P={step}: missing weight artifacts ({len(missing)} files).')
                continue
            weight_chunks = [_extract_weights_from_artifacts(group_dir, int(step)) for group_dir in group_dirs]
            weights = np.concatenate(weight_chunks, axis=0)
            num_members = int(weights.shape[0])
            sim = _weight_similarity_matrix(weights)
        elif ECDF_REPRESENTATION == 'activations':
            if probe_loader is None:
                probe_loader = _build_probe_loader(
                    probe_batch_size=ECDF_PROBE_BATCH_SIZE,
                    probe_seed=ECDF_PROBE_SEED,
                    probe_loader_batch_size=ECDF_PROBE_LOADER_BATCH_SIZE,
                )
            member_states = _collect_member_states(group_dirs, int(step), progress_label=f'ecdf w{ECDF_WIDTH} p{step}')
            num_members = len(member_states)
            sim = _activation_similarity_matrix(
                member_states,
                ECDF_WIDTH,
                probe_loader,
                progress_label=f'ecdf w{ECDF_WIDTH} p{step}',
            )
        else:
            raise ValueError("ECDF_REPRESENTATION must be 'weights' or 'activations'.")

        member_ids = build_member_ids(num_members, ECDF_WIDTH)
        within_real = extract_within_values(sim, member_ids)
        across_real = extract_across_values(sim, member_ids)

        npz_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            npz_path,
            within_real=np.asarray(within_real, dtype=np.float32),
            across_real=np.asarray(across_real, dtype=np.float32),
            width=np.int32(ECDF_WIDTH),
            images_seen=np.int64(step),
            representation=np.asarray(ECDF_REPRESENTATION),
        )
        print(f'Computed and saved {npz_path}')

    rng = np.random.default_rng(ECDF_RNG_SEED + int(step))
    within_real = _subsample(np.asarray(within_real), ECDF_MAX_POINTS, rng)
    across_real = _subsample(np.asarray(across_real), ECDF_MAX_POINTS, rng)

    ecdf_data[int(step)] = {
        'within_real': within_real,
        'across_real': across_real,
    }

if not ecdf_data:
    raise ValueError(
        f'No ECDF data prepared for width={ECDF_WIDTH}, representation={ECDF_REPRESENTATION}. '
        f'Tried {len(steps)} step(s); all were missing cache and could not be recomputed.'
    )

print(f'Prepared real ECDF distributions for {len(ecdf_data)} P values.')


In [ ]:
if not ecdf_data:
    raise ValueError('No ECDF data was prepared.')

steps_sorted = sorted(ecdf_data.keys())
fig, ax = plt.subplots(figsize=(12, 8))
cmap = plt.get_cmap('viridis')
den = max(len(steps_sorted) - 1, 1)

for idx, step in enumerate(steps_sorted):
    base_color = cmap(idx / den)
    pair_color = _lighten(base_color, amount=0.35)

    x_wr, y_wr = _ecdf_xy(ecdf_data[step]['within_real'])
    x_ar, y_ar = _ecdf_xy(ecdf_data[step]['across_real'])

    ax.plot(x_wr, y_wr, color=base_color, linewidth=2.0, linestyle='-', label=f'P={step} within_real')
    ax.plot(x_ar, y_ar, color=pair_color, linewidth=1.7, linestyle='-', label=f'P={step} across_real')

ax.set_xlabel('Absolute cosine similarity')
ax.set_ylabel('ECDF')
ax.set_title(f'ECDF by P (width={ECDF_WIDTH}, representation={ECDF_REPRESENTATION}, real only)')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()

ecdf_out = OUT_DIR / f'ecdf_similarity_real_by_p_w{ECDF_WIDTH}_{ECDF_REPRESENTATION}.pdf'
fig.savefig(ecdf_out, bbox_inches='tight')
display(fig)
plt.close(fig)
print(f'Wrote {ecdf_out}')
